# Safe-admission layer: easy and mixed scenarios

This notebook tests the **real active path** without PPO training:

`upper bias -> directional lower heuristic -> A3 eligibility -> safe admission -> successful handover -> load/reward`

It uses fresh environments and the same seed for every comparison. The safe-layer `accepted` counter is checked against successful handovers, so failed handovers must not consume quota.

## 1. Imports and configuration

- Easy scenario: one overloaded eMBB slice at gNB0.
- Mixed scenario: eMBB, URLLC, and mMTC overloaded at gNB0 and moving toward gNB1.
- The safe target-load limit is `0.80`.

In [ ]:
import os
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', '/tmp/mpl-safe-layer')
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from global_ppo_3gnb_env import GlobalPPO3GNBEnv, SLICE_TYPES

SEED = 123
EASY_SCENARIO = 'fixed_embb_g0_overlap'
MIXED_SCENARIO = 'mixed_g0_release'
SAFE_LIMIT = 0.80
print('Project root:', ROOT)
print('Slices:', SLICE_TYPES)

## 2. Experiment helpers

Each experiment starts from the same seed. One row represents one upper-control window. `quota` is shared across both neighbors for each source/slice.

In [ ]:
def make_env(scenario_name):
    return GlobalPPO3GNBEnv(
        seed=SEED,
        scenario_mode='curriculum',
        training_scenarios=scenario_name,
        scenario_selection='cycle',
        upper_window_seconds=1.0,
        local_steps_per_global=4,
        radio_substeps=2,
        terminal_reward_only=False,
        safe_admission_load_limits={s: SAFE_LIMIT for s in SLICE_TYPES},
    )


def matrix_columns(prefix, matrix):
    matrix = np.asarray(matrix, dtype=float)
    return {
        f'{prefix}_g{g}_{slice_type}': float(matrix[g, s])
        for g in range(3)
        for s, slice_type in enumerate(SLICE_TYPES)
    }


def readable_mapping(mapping):
    return {'/'.join(map(str, key)): int(value) for key, value in mapping.items()}


def run_experiment(label, scenario_name, bias_matrix, windows=1):
    env = make_env(scenario_name)
    rows = []
    try:
        _, reset_info = env.reset(seed=SEED)
        initial_loads = np.asarray(reset_info['load_matrix'], dtype=float)
        for window in range(windows):
            _, reward, terminated, truncated, info = env.step(
                np.asarray(bias_matrix, dtype=np.float32).reshape(-1)
            )
            safe = info['safe_admission']
            stats = safe['stats']
            successful = int(info['handover_count'])
            safe_accepted = int(stats['accepted'])
            assert safe_accepted == successful, (
                f'Quota accounting mismatch: accepted={safe_accepted}, '
                f'successful={successful}'
            )
            row = {
                'label': label,
                'scenario': scenario_name,
                'window': window,
                'reward': float(reward),
                'load_improvement': float(
                    info['load_imbalance_start'] - info['load_imbalance_end']
                ),
                'bias_smoothing_penalty': float(info['global_action_penalty']),
                'neutral_bias_penalty': float(info['reward_neutral_bias_penalty']),
                'wrong_bias_penalty': float(info['reward_wrong_bias_penalty']),
                'a3_eligible': int(stats['eligible']),
                'successful_handovers': successful,
                'safe_accepted': safe_accepted,
                'rejected_no_pressure': int(stats['rejected_no_pressure']),
                'rejected_no_source_excess': int(stats['rejected_no_source_excess']),
                'rejected_no_target_headroom': int(stats['rejected_no_target_headroom']),
                'rejected_capacity': int(stats['rejected_capacity']),
                'rejected_target_safety': int(stats['rejected_target_safety']),
                'rejected_step_budget': int(stats['rejected_step_budget']),
                'source_quota': readable_mapping(safe['source_capacities']),
                'source_accepted': readable_mapping(safe['source_accepted']),
                **matrix_columns('bias', bias_matrix),
                **matrix_columns('load', info['load_matrix']),
            }
            rows.append(row)
            if terminated or truncated:
                break
        return initial_loads, pd.DataFrame(rows)
    finally:
        env.close()


def plot_slice_loads(frame, title):
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
    x = frame['window'].to_numpy()
    for s, slice_type in enumerate(SLICE_TYPES):
        for g in range(3):
            axes[s].plot(x, frame[f'load_g{g}_{slice_type}'], marker='o', label=f'gNB{g}')
        axes[s].axhline(SAFE_LIMIT, color='red', linestyle='--', label='safe limit' if s == 0 else None)
        axes[s].set_title(slice_type)
        axes[s].set_xlabel('upper window')
        axes[s].grid(alpha=0.25)
    axes[0].set_ylabel('normalized slice load')
    axes[0].legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

## 3. Easy scenario: one overloaded eMBB source

Expected behavior:

- Neutral bias: no release pressure and no useful handover.
- `-0.2`: quota near 20% of the 10 source UEs.
- `-0.8`: larger quota, but radio/step limits may mean no additional successful handovers.
- `+0.8`: wrong retain action at the overloaded source.

In [ ]:
zero = np.zeros((3, 3), dtype=float)
easy_weak = zero.copy(); easy_weak[0, 0] = -0.2
easy_strong = zero.copy(); easy_strong[0, 0] = -0.8
easy_wrong_retain = zero.copy(); easy_wrong_retain[0, 0] = +0.8

easy_runs = []
for label, bias in [
    ('neutral', zero),
    ('weak_release_-0.2', easy_weak),
    ('strong_release_-0.8', easy_strong),
    ('wrong_retain_+0.8', easy_wrong_retain),
]:
    initial, frame = run_experiment(label, EASY_SCENARIO, bias, windows=1)
    easy_runs.append(frame)

easy_results = pd.concat(easy_runs, ignore_index=True)
print('Initial load matrix [gNB, slice]:')
display(pd.DataFrame(initial, index=['gNB0', 'gNB1', 'gNB2'], columns=SLICE_TYPES))
display(easy_results[[
    'label', 'reward', 'load_improvement', 'a3_eligible',
    'successful_handovers', 'safe_accepted', 'rejected_capacity',
    'rejected_target_safety', 'source_quota', 'source_accepted'
]])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
easy_results.plot.bar(x='label', y='reward', ax=axes[0], legend=False, color='tab:blue', title='Reward')
easy_results.plot.bar(x='label', y='successful_handovers', ax=axes[1], legend=False, color='tab:green', title='Successful HOs')
easy_results.plot.bar(x='label', y='rejected_capacity', ax=axes[2], legend=False, color='tab:orange', title='Rejected by quota')
for ax in axes:
    ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## 4. Mixed scenario: all three slices release from gNB0

The correct mixed action gives each overloaded gNB0 slice negative bias. The comparison uses a neutral action and an intentionally wrong positive-bias action. Twelve windows are simulated because this scenario includes movement and needs time to reach the handover region.

In [ ]:
mixed_release = np.zeros((3, 3), dtype=float)
mixed_release[0, :] = [-0.5, -0.5, -0.5]

mixed_wrong = np.zeros((3, 3), dtype=float)
mixed_wrong[0, :] = [+0.7, +0.7, +0.7]

mixed_runs = []
mixed_initial = {}
for label, bias in [
    ('neutral', zero),
    ('correct_mixed_release', mixed_release),
    ('wrong_mixed_retain', mixed_wrong),
]:
    initial, frame = run_experiment(label, MIXED_SCENARIO, bias, windows=12)
    mixed_initial[label] = initial
    mixed_runs.append(frame)

mixed_results = pd.concat(mixed_runs, ignore_index=True)
display(pd.DataFrame(
    mixed_initial['neutral'],
    index=['gNB0', 'gNB1', 'gNB2'],
    columns=SLICE_TYPES,
))

mixed_summary = mixed_results.groupby('label', as_index=False).agg(
    total_reward=('reward', 'sum'),
    total_load_improvement=('load_improvement', 'sum'),
    total_a3_eligible=('a3_eligible', 'sum'),
    total_successful_handovers=('successful_handovers', 'sum'),
    total_safe_accepted=('safe_accepted', 'sum'),
    rejected_capacity=('rejected_capacity', 'sum'),
    rejected_target_safety=('rejected_target_safety', 'sum'),
    rejected_no_pressure=('rejected_no_pressure', 'sum'),
)
display(mixed_summary)

In [ ]:
for label in mixed_results['label'].unique():
    plot_slice_loads(
        mixed_results[mixed_results['label'] == label],
        f'{MIXED_SCENARIO}: {label}',
    )

## 5. Rejection-reason inspection

This table helps distinguish lower-layer radio limitations from safe-layer blocking:

- `a3_eligible = 0`: no radio-eligible candidate reached the safe layer.
- `rejected_no_pressure`: bias was zero or positive.
- `rejected_no_source_excess`: legacy counter; the safe layer no longer cancels a negative-bias request based on source-average load.
- `rejected_capacity`: the shared source quota was exhausted.
- `rejected_target_safety`: receiving the UE would exceed the safe target-load limit.
- `rejected_step_budget`: more candidates existed than the per-local-step handover limit.

In [ ]:
rejection_columns = [
    'label', 'window', 'a3_eligible', 'successful_handovers',
    'rejected_no_pressure', 'rejected_no_source_excess',
    'rejected_no_target_headroom', 'rejected_capacity',
    'rejected_target_safety', 'rejected_step_budget',
    'source_quota', 'source_accepted',
]
display(mixed_results[rejection_columns])

## 6. Direct good-action versus bad-action comparison

This section is specifically for checking the learning signal. Every action is evaluated from a fresh environment with the same seed.

- **Good selective release:** negative bias only on the overloaded source slices.
- **Bad retain:** positive bias on overloaded source slices, preventing useful release.
- **Bad release everywhere:** strong negative bias on every gNB and slice, including cells that should not release.

A useful reward design should rank the selective action above both bad actions.

In [ ]:
def final_load_imbalance(frame):
    last = frame.iloc[-1]
    loads = np.asarray([
        [last[f'load_g{g}_{slice_type}'] for slice_type in SLICE_TYPES]
        for g in range(3)
    ], dtype=float)
    return float(np.sum((loads - loads.mean(axis=0, keepdims=True)) ** 2))


def compare_actions(scenario_name, action_map, windows):
    rows = []
    traces = {}
    for action_name, action in action_map.items():
        _, frame = run_experiment(action_name, scenario_name, action, windows=windows)
        traces[action_name] = frame
        rows.append({
            'scenario': scenario_name,
            'action': action_name,
            'episode_reward': float(frame['reward'].sum()),
            'load_improvement': float(frame['load_improvement'].sum()),
            'final_load_imbalance': final_load_imbalance(frame),
            'successful_handovers': int(frame['successful_handovers'].sum()),
            'safe_accepted': int(frame['safe_accepted'].sum()),
            'a3_eligible': int(frame['a3_eligible'].sum()),
            'rejected_capacity': int(frame['rejected_capacity'].sum()),
            'rejected_target_safety': int(frame['rejected_target_safety'].sum()),
            'wrong_bias_penalty': float(frame['wrong_bias_penalty'].sum()),
        })
    return pd.DataFrame(rows), traces


easy_good = np.zeros((3, 3)); easy_good[0, 0] = -0.2
easy_bad_retain = np.zeros((3, 3)); easy_bad_retain[0, 0] = +0.8
bad_release_everywhere = np.full((3, 3), -0.8)

mixed_good = np.zeros((3, 3)); mixed_good[0, :] = -0.5
mixed_bad_retain = np.zeros((3, 3)); mixed_bad_retain[0, :] = +0.8

easy_comparison, easy_comparison_traces = compare_actions(
    EASY_SCENARIO,
    {
        'GOOD_selective_release': easy_good,
        'BAD_retain_overloaded_source': easy_bad_retain,
        'BAD_release_everywhere': bad_release_everywhere,
    },
    windows=12,
)

mixed_comparison, mixed_comparison_traces = compare_actions(
    MIXED_SCENARIO,
    {
        'GOOD_selective_release': mixed_good,
        'BAD_retain_overloaded_source': mixed_bad_retain,
        'BAD_release_everywhere': bad_release_everywhere,
    },
    windows=12,
)

action_comparison = pd.concat([easy_comparison, mixed_comparison], ignore_index=True)
display(action_comparison.sort_values(['scenario', 'episode_reward'], ascending=[True, False]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, (scenario, group) in zip(axes, action_comparison.groupby('scenario')):
    colors = ['tab:green' if name.startswith('GOOD') else 'tab:red' for name in group['action']]
    ax.bar(group['action'], group['episode_reward'], color=colors)
    ax.axhline(0.0, color='black', linewidth=1)
    ax.set_title(f'{scenario}: reward comparison')
    ax.set_ylabel('sum of upper-window rewards')
    ax.tick_params(axis='x', rotation=25)
    ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

for scenario, group in action_comparison.groupby('scenario'):
    ranked = group.sort_values('episode_reward', ascending=False)
    print(f'\n{scenario} reward ranking:')
    display(ranked[['action', 'episode_reward', 'load_improvement',
                    'final_load_imbalance', 'successful_handovers',
                    'wrong_bias_penalty', 'rejected_target_safety']])
    good_reward = float(group.loc[group['action'] == 'GOOD_selective_release', 'episode_reward'].iloc[0])
    best_bad_reward = float(group.loc[group['action'].str.startswith('BAD'), 'episode_reward'].max())
    if good_reward <= best_bad_reward:
        print('WARNING: a bad action scores as well as or better than the selective action.')
        print('The safe layer may still be shielding bad upper intent.')
    else:
        print('PASS: the selective good action scores above both bad actions.')

## 7. Automatic sanity checks

These assertions verify the corrected accounting rule and basic safe-layer semantics.

In [ ]:
assert (easy_results['safe_accepted'] == easy_results['successful_handovers']).all()
assert (mixed_results['safe_accepted'] == mixed_results['successful_handovers']).all()

easy_by_label = easy_results.set_index('label')
assert easy_by_label.loc['weak_release_-0.2', 'reward'] > easy_by_label.loc['wrong_retain_+0.8', 'reward']
assert easy_by_label.loc['wrong_retain_+0.8', 'successful_handovers'] == 0
mixed_by_label = mixed_summary.set_index('label')
assert mixed_by_label.loc['correct_mixed_release', 'total_successful_handovers'] > 0
assert mixed_by_label.loc['correct_mixed_release', 'total_safe_accepted'] == mixed_by_label.loc['correct_mixed_release', 'total_successful_handovers']
assert mixed_by_label.loc['wrong_mixed_retain', 'total_successful_handovers'] == 0
for scenario_name, group in action_comparison.groupby('scenario'):
    good_reward = float(group.loc[group['action'] == 'GOOD_selective_release', 'episode_reward'].iloc[0])
    best_bad_reward = float(group.loc[group['action'].str.startswith('BAD'), 'episode_reward'].max())
    assert good_reward > best_bad_reward, (scenario_name, good_reward, best_bad_reward)

print('PASS: safe accepted counters equal successful handovers.')
print('PASS: useful easy-scenario release beats the wrong retain action.')
print('PASS: positive source bias does not admit outgoing handovers.')
print('PASS: the mixed release case exercises real safe-admitted handovers.')
print('PASS: selective release beats every bad action in both scenarios.')